### 06 CNN-Modell

***Hinweis:*** Führen Sie dieses Notebook aus, nachdem Sie das `05 Random Forest-Modell` Notebook ausgeführt haben, um die erforderlichen Metadaten zu generieren.

#### CNN-Modelltraining

In diesem Notebook widmen wir uns der extraktiven Analyse bildbasierter Merkmale von Instagram-Posts mittels eines Convolutional Neural Networks (CNN), um den Einfluss visueller Eigenschaften auf die Anzahl der Likes zu untersuchen. Hierzu setzen wir auf Transfer Learning mit einem vortrainierten ResNet50–Modell, das durch seine 50 Residual-Schichten eine ausgewogene Kombination aus Modelltiefe und Trainingsstabilität bietet und dank der ImageNet-Gewichte allgemeine visuelle Muster (z. B. Kanten, Formen, Farbkontraste) effektiv erfasst. Nach Entfernen des Top-Klassifikators extrahieren wir aus jedem Instagram-Bild kompakte Feature-Vektoren, die als Grundlage für unsere anschließende Regression dienen.

**Ziel und Vorgehen:**

* Visuelle Merkmale (Merkmalsvektoren) aus Bilddaten extrahieren (mit CNN – ResNet50)
* Regressionsmodell auf Basis der CNN-Features (einfaches Feedforward-Regressionsnetzwerk trainieren)
* Logarithmierte Like-Zahl (`likes_log`) vorhersagen
* Bewertung der Vorhersagegüte mittels mittlerem quadratischem Fehler (MSE) und Bestimmtheitsmaß (R²)

**Hypothese (visuelle Merkmale):**  
- **H0:** Die visuellen Merkmale, extrahiert durch ResNet50, haben keinen linearen Einfluss auf `likes_log`.  
- **H1:** Mindestens ein visueller Merkmalsvektor-Eintrag beeinflusst signifikant `likes_log`.

**Hinweis:**
Videos werden in diesem Modell nicht berücksichtigt. Das Projekt beschränkt sich in der Analyse zur Vorhersage auf Bildformate und die enthaltenden Metadaten.

In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
import tensorflow as tf
import threadpoolctl

from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, Input, GlobalAveragePooling2D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:
base_dir = Path.cwd()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent
    os.chdir(base_dir)

#### Datenvorbereitung
**Relevante Metadaten laden** (likes_log und post_id)

In [ ]:
df_meta = pd.read_pickle('ergebnisse/verarbeitete_daten.pkl')
print(f"Metadaten geladen: {df_meta.shape[0]} Zeilen, {df_meta.shape[1]} Spalten")

print("Spalten von df_meta:")
print(df_meta.columns.tolist())

df_meta.head(3) 



Für das Laden von Bildern und Zuordnen der Posts, ist eine eindeutige Identifikation der Posts essentiell, sodass nun die ursprüngliche Spalte folder_name, wie zuvor in der Datenvorbereitung definiert, nun folgend als post_id Bedeutung erhält.



**Bilder laden:**

Die Datensätze liegen als Rohdaten vor: Pro Post ein Ordner mit Bilddateien. Videos (Datei-Endungen mit .mp4) werden nicht berücksichtigt.

In [ ]:
image_base_dir = "daten/rohdaten/instagram-posts-dataset/Data"

if not os.path.isdir(image_base_dir):
    raise FileNotFoundError(
        f"Bildverzeichnis '{image_base_dir}' nicht gefunden. "
        "Bitte überprüfe, ob der Datensatz korrekt heruntergeladen wurde."
    )
else:
    print(f"Bildverzeichnis gefunden: {image_base_dir}")


# Einsammeln aller Bilddateien rekursiv (jpg, jpeg, png)  
image_extensions = ("*.jpg", "*.jpeg", "*.png")
all_image_paths = []
for ext in image_extensions:
    pattern = os.path.join(image_base_dir, "**", ext)
    all_image_paths.extend(glob.glob(pattern, recursive=True))

if len(all_image_paths) == 0:
    raise RuntimeError("Keine Bilddateien gefunden. Stelle sicher, dass im Verzeichnis Bilder liegen.")
else:
    print(f"Gefundene Bilder: {len(all_image_paths)}")

def extract_post_id_from_path(img_path):
    parts = img_path.split(os.sep)
    post_id = parts[-2]
    return post_id

df_images = pd.DataFrame({
    "image_path": all_image_paths,
    "post_id": [extract_post_id_from_path(p) for p in all_image_paths]
})

print("Beispielhafte post_id‐Werte in df_images:")
print(df_images["post_id"].unique()[:10])
print(f"Anzahl eindeutiger post_id in df_images: {df_images['post_id'].nunique()}")

print("Beispiel Post-ID-Extraktion aus Bildpfaden:")
display(df_images.head(5))


**Mapping von Bildpfad auf Post-ID**

Die Ordnerstruktur ist so, dass jedes Bild im Unterordner {post_id} liegt. Es wird eine Funktion definiert, um aus dem Dateipfad die post_id zu extrahieren. Es werden alle Bilddateien innerhalb des Ordners gesammelt.


In [ ]:
def extract_post_id_from_path(img_path):  # nimmt an, dass in dem Ordner-Layout einen eigenen Unterordner zu jedem Post gibt und Bilddateien mit Endung .jpg
                                          # Beispiel: "/…/Data/diipakhosla_1908495_2995911890106994559_10447_62/DiipaKhosla_1908495_2995911890106994559_10447_62_1.jpg"
    parts = img_path.split(os.sep)
    # Die vorletzte Ebene ist der Post-Ordner
    post_id = parts[-2]
    return post_id

# Erstellen einer DataFrame-Tabelle mit Spalten: image_path, post_id
df_images = pd.DataFrame({
    "image_path": all_image_paths,
    "post_id": [extract_post_id_from_path(p) for p in all_image_paths]
})

# Test: Anzahl Bilder pro Post anzeigen 
print("Beispiel: Bilder pro Post-ID")
print(df_images.groupby("post_id").size().head(5))


**Metadaten und Bildpfade verknüpfen**

Es soll ein Inner-Join durchgeführt werden, sodass nur Bilder übrigbleiben, deren Post-ID in den Metadaten vorhanden ist, sodass diese verknüpft werden können.

In [ ]:
# Metadaten und Bildpfade verknüpfen

#*Überprüfen, ob die Metadaten eine Spalte post_id haben: 
#    - Automatisches Suchen nach einer Spalte in df_meta, die mit den post_id-Werten aus df_images übereinstimmt
#    - Anschließend soll ein Inner-Join durchgeführt werden, sodass nur Bilder übrigbleiben, deren Post-ID in den Metadaten vorhanden ist.

candidate_cols = [c for c in df_meta.columns if "post" in c.lower() and "id" in c.lower()]
print(f"Potenzielle Post-ID-Spalten in Metadaten: {candidate_cols}")

# Prüfen, ob eine dieser Spalten alle IDs aus df_images enthält
found_post_id_col = None
for col in candidate_cols:# 
    meta_set = set(df_meta[col].astype(str).unique())
    img_set = set(df_images["post_id"].unique())
    # Wenn alle post_id aus Bildern in Metadaten vorkommen:
    if img_set.issubset(meta_set):
        found_post_id_col = col
        break
      
if found_post_id_col is None:  # manuelle Festlegung der Spalte nach Prüfung, welche Spalte wirklich passt
    found_post_id_col = "post_id"        
    print("Automatische Suche hat keine Spalte gefunden.")
    print(f"Verwende jetzt manuell: '{found_post_id_col}'")
else:
    print(f"Automatisch ermittelte Post-ID-Spalte: '{found_post_id_col}'")


**Merge der Bilder mit den Metadaten**

Eine muss entsprechende Spalte als Fremdschlüssel (post_id) in den Metadaten existieren, um diese zusammenzuführen. Sodass
ein Datatframe pro Bild alle relevanten Metadaten erhält inklusive der Zielvariable likes_log.

In [ ]:
# Merge der Bilder mit den Metadaten
# Damit die Bilder mit den Metadaten verknüpft werden können, muss eine entsprechende Spalte als Fremdschlüssel (post_id) in den Metadaten existieren, um diese zusammenzuführen. Sodass
# - nur Bilder übrig bleiben, deren post_id in den Metadaten existiert und
# - df_merged pro Bild alle relevanten Metadaten enthält, inklusive der Zielvariable likes_log.
# - Konvertieren der ID-Spalte in String (falls nicht schon String)

df_meta[found_post_id_col] = df_meta[found_post_id_col].astype(str)

# Merge auf 'post_id' im Bild-DF und der entsprechenden Spalte in Metadaten
df_merged = df_images.merge(
    df_meta,
    left_on="post_id",
    right_on=found_post_id_col,
    how="inner"
)

print(f"Nach Merge: {df_merged.shape[0]} kombinierte Zeilen (Bild + Metadaten).")
df_merged.head(5)


**Bildvorverarbeitung: Parameter und Hilfsfunktionen**

Jedes Bild wird in 224×224×3 skaliert und Funktionen der tensorflow.keras.applications.resnet50 verwendet, um dieselbe Vorverarbeitung wie beim ImageNet-Training anzuwenden (Subtrahieren von Mittelwerten, Skalierung etc.).

In [ ]:
# Bildgröße für ResNet50
IMG_SIZE = (224, 224)

# Hilfsfunktion: Ein einzelnes Bild laden und auf 224x224 (IMG_SIZE) skalieren, Durchführung ResNet50-spezifische Vorverarbeitung
def load_and_preprocess(img_path):
    img = load_img(img_path, target_size=IMG_SIZE)    # 1. Bild als PIL-Objekt laden
    arr = img_to_array(img)                           # 2. Zu Array (H, W, 3) konvertieren
    arr = preprocess_input(arr)                       # 3. Vorverarbeitung (Subtrahiere Imagenet-Mittelwerte usw.)
    return arr

# Beispiel: Test-Ladefunktion
test_img_path = df_merged["image_path"].iloc[0]
test_arr = load_and_preprocess(test_img_path)
print(f"Form des Test-Arrays: {test_arr.shape}  (Höhe, Breite, Kanäle)")


#### Feature-Extraktion mit ResNet50

* Laden des vortrainierten ResNet50-Modell (Gewichte: ImageNet) ohne den finalen Dense-Block
* Verwendung von GlobalAveragePooling2D, um aus den letzten Convolution-Maps einen 2048-dimensionalen Vektor zu erzeugen.
* Batch-Verarbeitung reduziert Speicherbedarf und erhöht Effizienz.

In [ ]:
# ResNet50 ohne Top-Layer laden

# ResNet50-Basis laden: weights='imagenet', ohne das Top (Dense-) Layer, (include_top=False)
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

x = base_model.output               # Output der letzten Convolution-Maps 
x = GlobalAveragePooling2D()(x)     # Alternativ: Auf GlobalAveragePooling-Ebene greifen

# Definieren eines Model-Objekts, das Bild Input als 2048-dimensionaler Feature-Vektor ausgibt
model_resnet_feat = Model(inputs=base_model.input, outputs=x)

print("ResNet50-Feature-Extractor geladen.")
model_resnet_feat.summary()


##### Extrahiere Features für alle Bilder (Batch-Verarbeitung)

Erzeugung einwa großes NumPy-Arraya mit Form (n_images, 2048). Um Speicher zu sparen, werden die Batches iteriert.

* GlobalAveragePooling2D soll die räumlichen Dimensionen reduzieren und einen festen 2048-dimensionalen Vektor zurückgeben
* Die Batch-Verarbeitung (z. B. 32 Bilder pro Batch) soll den GPU-/RAM-Speicher schonen

In [ ]:
# Anzahl Bilder
n_images = df_merged.shape[0]
print(f"Insgesamt {n_images} Bilder → Extrahiere Features...")

# Speichern für Feature-Vektoren, ResNet50 GlobalAveragePooling erzeugt 2048-D-Vektor
features = np.zeros((n_images, 2048), dtype=np.float32)

# Iterieren über alle Bilder in Chargen (Batch-Größe ≈ 32)
batch_size = 32
for start in range(0, n_images, batch_size):
    end = min(start + batch_size, n_images)
    batch_paths = df_merged["image_path"].iloc[start:end].tolist()
    batch_arrays = np.zeros((len(batch_paths), 224, 224, 3), dtype=np.float32)  # Erstellt leeren Batch-Array
    
    for i, p in enumerate(batch_paths):   # Laden und vorbereiten jedes Bildes im Batch
        try:
            batch_arrays[i] = load_and_preprocess(p)
        except Exception as e:
            print(f"Warnung: Fehler beim Laden von {p}: {e}")
            batch_arrays[i] = np.zeros((IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)   # Bei Fehler Array auf Null setzen (oder überspringen)
    
    # Extrahieren der Features (Batch-Durchlauf durch ResNet50)
    batch_feats = model_resnet_feat.predict(batch_arrays, verbose=0)
    features[start:end, :] = batch_feats

print("Feature-Extraktion abgeschlossen.")
print(f"Shape des Feature-Arrays: {features.shape}")


#### Dataset vorbereiten für Regression

Genutzt werden hier 60 % der Bilder zum Training und 40 % zum Testen. Die Split-Ratio 60/40 ermöglicht Vergleichbarkeit mit dem OLS- und RF-Modell.

* Zielvariable (likes_log) extrahieren
* Train/Test-Split

In [ ]:
# Zielvariable y (likes_log) extrahieren, in df_merged muss es eine Spalte 'likes_log' geben 
if "likes_log" not in df_merged.columns:
    raise KeyError("Spalte 'likes_log' nicht in den Metadaten gefunden.")

y = df_merged["likes_log"].values
print(f"Zielvariable y (likes_log) geladen: {y.shape[0]} Werte")

# Train/Test-Split
X_train_feat, X_test_feat, y_train, y_test = train_test_split(
    features, y, test_size=0.40, random_state=42
)

print(f"Trainingsdaten: X={X_train_feat.shape}, y={y_train.shape}")
print(f"Testdaten:       X={X_test_feat.shape}, y={y_test.shape}")

#### Speichern der Bildmerkmale in einer CSV-Datei

Die extrahierten 2048-dimensionalen Feature-Vektoren sollen zusammen mit den image_path, post_id und likes_log in einer CSV-Datei ablegen.
Dateipfad: ~/daten/verarbeitete_daten/bild_merkmale_likes.csv

Mit to_csv(index=False) werden die Daten ohne zusätzlichen Index gespeichert, sodass die CSV für weitere Analysen verwendet und zur Bewertung herangezogen werden kann.

In [ ]:
# DataFrame für alle Bilder vor Split (df_merged enthält Zeilen in derselben Reihenfolge wie `features`)
df_features = pd.DataFrame(
    data=features,
    columns=[f"feat_{i}" for i in range(features.shape[1])]
)

# 'image_path', 'post_id' und 'likes_log' hinzufügen
df_features.insert(0, "likes_log", df_merged["likes_log"].values)
df_features.insert(0, "post_id", df_merged["post_id"].values)
df_features.insert(0, "image_path", df_merged["image_path"].values)

print(f"DataFrame mit Bildmerkmalen: {df_features.shape} (Zeilen, Spalten)")
df_features.head(3)


# Speicherort der CSV
output_dir = "daten/verarbeitete_daten"
output_csv = os.path.join(output_dir, "bild_merkmale_likes.csv")

# Erstelle Verzeichnis, falls es nicht existiert
os.makedirs(output_dir, exist_ok=True)

print(f"CSV-Datei wird gespeichert unter: {output_csv}")


# Speichere die Zeilen 
df_features.to_csv(output_csv, index=False)
print("Bildmerkmale wurden erfolgreich in CSV gespeichert.")



#### Modellarchitektur & Training

Ziel: 
1. Einfaches Dense-Regressionsnetz definieren und trainieren
2. Training CNN-Regressors

##### Einfaches Dense-Regressionsnetz

* Zwei vollverknüpfte („Dense“) Schichten mit ReLU und Dropout (0.5) zur Regularisierung
* Letzte Schicht: Linearer Knoten ->  Regressionsvorhersage (likes_log)

In [ ]:
# Eingabe ist der 2048-D-Feature-Vektor von ResNet50
input_dim = X_train_feat.shape[1]  # 2048

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="linear")   # Regressionwert likes_log
])

# Kompilieren
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="mean_squared_error",
    metrics=[]
)

model.summary()


##### Training starten

* Wir trainieren 20 Epochen mit Batch-Größe 32
* 20 % des Trainingssets nutzen als interne Validierung, um Overfitting zu beobachten

In [ ]:
# Hyperparameter
EPOCHS = 20
BATCH_SIZE = 32

history = model.fit(
    X_train_feat, y_train,
    validation_split=0.2,  # 20 % der Trainingsdaten als Validierung
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


#### Evaluation des CNN-Modells

- Training‐ und Validierungsverlauf (Lernkurven)
- MSE & R² auf Testset berechnen
- Scatterplot: Echte vs. Prognostizierte Werte

In [ ]:
# Training‐ und Validierungsverlauf (Lernkurven)
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoche")
plt.ylabel("MSE Loss")
plt.title("Lernkurven des CNN-Regressionsmodells")
plt.legend()
plt.show()

# MSE & R² auf Testset berechnen
y_pred_test = model.predict(X_test_feat).flatten()

mse_cnn = mean_squared_error(y_test, y_pred_test)
r2_cnn  = r2_score(y_test, y_pred_test)

print(f"Test MSE (CNN-Modell): {mse_cnn:.4f}")
print(f"Test R² (CNN-Modell): {r2_cnn:.4f}")

# Scatterplot: Echte vs. Prognostizierte Werte
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test, alpha=0.4, edgecolors='k', s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Echte likes_log")
plt.ylabel("Prediktierte likes_log")
plt.title("CNN: Echte vs. vorhergesagte Werte")
plt.tight_layout()
plt.show()

**Ergebnisse:** (ERGÄNZUNGEN!)

* MSE (Mean Squared Error) und R² (Bestimmtheitsmaß) quantifizieren die Performance.
* Scatterplot zeigt, wie gut die Punkte um die Diagonale liegen.

#### Keras-Modell und Vorhersagen speichern

Damit die CNN-Vorhersagen später für weitere Auswertungen verwendet werden können, werden sie als pkl-Datei unter ~/modelle gespeichert.

Zusätzlich wird das Keras-Modell samt Architektur und Gewichtung dort festgehalten.

In [ ]:
# Erstelle Ordner „modelle“ unter „ergebnisse/modelle“, falls nicht vorhanden
model_dir = "ergebnisse/modelle"
os.makedirs(model_dir, exist_ok=True)

# Speichern des gesamten Keras-Modells (.h5)
keras_model_path = os.path.join(model_dir, "like_vorhersage_modell.h5")
model.save(keras_model_path)
print(f"Keras-Modell gespeichert: {keras_model_path}")

# Pickle der Vorhersage-Arrays (cnn_pred_train, cnn_pred_test)
cnn_pred_train = model.predict(X_train_feat).flatten()
cnn_pred_test  = y_pred_test  # schon berechnet

pickle_path = os.path.join(model_dir, "like_vorhersage_modell.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump({
        "cnn_pred_train": cnn_pred_train,
        "cnn_pred_test": cnn_pred_test
    }, f)

print(f"CNN-Vorhersagen gespeichert als Pickle: {pickle_path}")
